# 02 — Diagnóstico de Esquemas y Preparación de Reconstrucción

Este notebook corresponde a la **sección de Persona 1** del diagnóstico. Compara esquemas reales contra esquemas esperados y deja preparados los JSON de metadatos para Persona 2.

In [1]:
!pip install -q pyspark pyyaml
!pip install -q pyarrow



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import json
from pathlib import Path

# Ruta del proyecto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Agregar carpeta src al path
sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

# Configuración necesaria para PySpark en Windows
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Proyecto:", PROJECT_ROOT)
print("Python usado por Spark:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

Proyecto: c:\Modelado\ProyectoSegundoParcial
Python usado por Spark: c:\Users\HP OMEN\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


In [3]:
import importlib
import schema_recovery
importlib.reload(schema_recovery)
from schema_recovery import diagnose_successful_files, write_schema_differences
from utils import load_config, get_spark_session

config = load_config("config/etl_config.yaml")
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")


## 1. Cargar inventario técnico generado 

In [4]:
import pandas as pd
from pyspark.sql import functions as F

audit_path = Path(config["paths"]["audit"]) / "audit_file_inventory"
inventory_pd = pd.read_parquet(str(audit_path), engine="pyarrow")

inventory_str = inventory_pd.astype(str).replace(
    {"nan": None, "<NA>": None, "NaT": None, "None": None, "": None}
)

inventory_df = spark.createDataFrame(inventory_str)

# try_cast devuelve NULL en vez de error para NaN/valores inválidos
inventory_df = (inventory_df
    .withColumn("file_size_mb",    F.expr("try_cast(file_size_mb as double)"))
    .withColumn("partition_year",  F.expr("try_cast(try_cast(partition_year as double) as bigint)"))
    .withColumn("partition_month", F.expr("try_cast(try_cast(partition_month as double) as bigint)"))
    .withColumn("record_count",    F.expr("try_cast(try_cast(record_count as double) as bigint)"))
    .withColumn("column_count",    F.expr("try_cast(try_cast(column_count as double) as bigint)"))
)

inventory_df.show(30, truncate=80)


+------------------------------------+----------------------+------------+-----------------------------------+--------------------------------------------------------------------------------+------------+----------------------------------------------------------------+--------------+---------------+------------+------------+------------+----------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------+
|                          process_id|         source_system|service_type|                          file_name|                                                                       file_path|file_size_mb|                                                file_hash_sha256|partition_year|partition_month| read_status|record_count|column_count|                                                     schema_hash|                                                                   error_message|  

## 2. Mostrar esquemas esperados creados en metadata/

In [5]:
metadata_files = [
    "metadata/expected_schema_yellow.json",
    "metadata/expected_schema_green.json",
    "metadata/expected_schema_fhvhv.json",
    "metadata/canonical_schema_trips.json",
    "metadata/business_rules.json",
]

for file in metadata_files:
    print("\n===", file, "===")
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])


=== metadata/expected_schema_yellow.json ===
{
  "service_type": "yellow",
  "description": "Esquema esperado aproximado para archivos NYC TLC Yellow Taxi Trip Data.",
  "fields": [
    {
      "name": "VendorID",
      "type": "long",
      "nullable": true
    },
    {
      "name": "tpep_pickup_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "tpep_dropoff_datetime",
      "type": "timestamp",
      "nullable": true
    },
    {
      "name": "passenger_count",
      "type": "double",
      "nullable": true
    },
    {
      "name": "trip_distance",
      "type": "double",
      "nullable": true
    },
    {
      "name": "RatecodeID",
      "type": "double",
      "nullable": true
    },
    {
      "name": "store_and_fwd_flag",
      "type": "string",
      "nullable": true
    },
    {
      "name": "PULocationID",
      "type": "integer",
      "nullable": true
    },
    {
      "name": "DOLocationID",
      "type": "integer",
      "nul

## 3. Comparación de esquema real vs esquema esperado

Se generan tres tipos de diferencia: `EXTRA_COLUMN`, `MISSING_COLUMN` y `TYPE_MISMATCH`.

In [6]:
df_check = inventory_df.toPandas()
print("Total filas:", len(df_check))
print("Valores read_status:", df_check["read_status"].value_counts().to_dict())


Total filas: 20
Valores read_status: {'SUCCESS': 18, 'SCHEMA_ERROR': 1, 'CORRUPT': 1}


In [7]:
df_inv = inventory_df.toPandas()
result = (df_inv[df_inv["read_status"] == "SUCCESS"]
    .groupby(["service_type", "schema_hash"])
    .size()
    .reset_index(name="count")
    .sort_values(["service_type", "count"]))
print(result.to_string(index=False))


service_type                                                      schema_hash  count
 bad_parquet 145ef2eda729c78f944e6b1bfe4fb8a076f2222eee59c616f996e38e06fd726a      1
 bad_parquet 2ee393a89b3ae9815f3e24f047f03d2a0fcf69d059e4bd8930290830c43de52c      1
 bad_parquet 7e3103878695cb996098b571752d1ebf746ae6c1366189cfbbc50eaa9acc2798      1
 bad_parquet 89ae3cf20730e6fa8d84afe41367821a61036f5bbd280705834456960a506253      1
 bad_parquet f98ab85e87dd0b98219c10b071b90b17e318fe6bd3aa1268e055d8dcf8331263      1
 bad_parquet 92681c0c3ddf2795b6c197dcddf2b9a13a51f824d0ba378d753ae18e969d0a66      2
       fhvhv db4cd5c4701313fb72a9082181dc2497a3df672bd4820d94e544377d0440252e      1
       green 003d7291ad2ae7399c90f5da09fed57bf9493d755058ef9ab946e8ef7317b823      1
       green 3a7c6d677e6b981a890fd8799a839b2ad128093013f3e4abf2d714f09e92ec94      1
      yellow 9c74eb92d675451104d7898d717cc313b0e0150024da7a1c4dd88034a72b4d8e      1
      yellow f8ace373f12c24ff3bdccfa9da15cb3415082626673c98d1aa01

In [8]:
import importlib
import schema_recovery
importlib.reload(schema_recovery)
from schema_recovery import diagnose_successful_files, write_schema_differences

# Calcula schema_diff_df si no está definido (ejecución de celda individual)
if "schema_diff_df" not in globals():
    process_id = inventory_df.select("process_id").first()[0]
    schema_diff_df = diagnose_successful_files(spark, config, inventory_df, process_id)
    print(f"schema_diff_df calculado: {schema_diff_df.count()} diferencias")

schema_diff_path = write_schema_differences(schema_diff_df, config)
print("schema_differences escrito en:", schema_diff_path)


schema_diff_df calculado: 36 diferencias
schema_differences escrito en: C:\Modelado\ProyectoSegundoParcial\data\audit\schema_differences


## 4. Resumen de diferencias por servicio

In [9]:
# Guard: recalcular schema_diff_df si no está en contexto
if "schema_diff_df" not in globals():
    import importlib, schema_recovery
    importlib.reload(schema_recovery)
    from schema_recovery import diagnose_successful_files
    process_id = inventory_df.select("process_id").first()[0]
    schema_diff_df = diagnose_successful_files(spark, config, inventory_df, process_id)

df_counts = (schema_diff_df.toPandas()
    .groupby(["service_type", "diff_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["service_type", "diff_type"]))
print(df_counts.to_string(index=False))


service_type      diff_type  count
       fhvhv  TYPE_MISMATCH      2
       green  TYPE_MISMATCH      8
      yellow   EXTRA_COLUMN      3
      yellow MISSING_COLUMN      3
      yellow  TYPE_MISMATCH     20


## 5. Hashes de esquema por servicio

Permite identificar cambios estructurales entre meses o versiones del mismo tipo de servicio.

In [10]:
inventory_df.filter("read_status = 'SUCCESS'").groupBy(
    "service_type", "schema_hash"
).count().orderBy("service_type", "count").show(100, truncate=False)

+------------+----------------------------------------------------------------+-----+
|service_type|schema_hash                                                     |count|
+------------+----------------------------------------------------------------+-----+
|bad_parquet |89ae3cf20730e6fa8d84afe41367821a61036f5bbd280705834456960a506253|1    |
|bad_parquet |f98ab85e87dd0b98219c10b071b90b17e318fe6bd3aa1268e055d8dcf8331263|1    |
|bad_parquet |145ef2eda729c78f944e6b1bfe4fb8a076f2222eee59c616f996e38e06fd726a|1    |
|bad_parquet |7e3103878695cb996098b571752d1ebf746ae6c1366189cfbbc50eaa9acc2798|1    |
|bad_parquet |2ee393a89b3ae9815f3e24f047f03d2a0fcf69d059e4bd8930290830c43de52c|1    |
|bad_parquet |92681c0c3ddf2795b6c197dcddf2b9a13a51f824d0ba378d753ae18e969d0a66|2    |
|fhvhv       |db4cd5c4701313fb72a9082181dc2497a3df672bd4820d94e544377d0440252e|1    |
|green       |003d7291ad2ae7399c90f5da09fed57bf9493d755058ef9ab946e8ef7317b823|1    |
|green       |3a7c6d677e6b981a890fd8799a839b2ad1280930

---

## Sección 2 — Reconstrucción del Esquema Canónico 

Esta sección toma los DataFrames exitosamente leídos y los homologa al **esquema canónico de 24 campos** definido en `canonical_schema_trips.json`, usando las reglas de `business_rules.json`.

Pasos:
1. Importar funciones 
2. Aplicar `apply_canonical_schema()` a cada archivo SUCCESS por servicio
3. Clasificar archivos dañados y generar registros de quarantine
4. Unir los DataFrames canónicos y escribir en `data/bronze/`

In [11]:
from schema_recovery import (
    apply_canonical_schema,
    classify_corrupt_file,
    generate_quarantine_record,
    CANONICAL_FIELDS,
)

print("Funciones de Persona 2 importadas correctamente.")
print("Campos canonicos:", CANONICAL_FIELDS)

Funciones de Persona 2 importadas correctamente.
Campos canonicos: ['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'improvement_surcharge', 'quality_status']


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Aplicar esquema canonico a cada archivo SUCCESS por servicio
# Resultado: un DataFrame unificado por servicio con exactamente 24 campos

canonical_dfs = {}

for service in ["yellow", "green", "fhvhv"]:
    success_files = (
        inventory_df
        .filter(f"read_status = 'SUCCESS' AND service_type = '{service}'")
        .select("file_path", "file_name")
        .collect()
    )

    service_parts = []
    n_rows_total = 0
    for row in success_files:
        with open(row["file_path"], "rb") as _fh:
            _df_pd = pd.read_parquet(_fh, engine="pyarrow")
        n_rows = len(_df_pd)
        n_rows_total += n_rows
        df_raw = spark.createDataFrame(_df_pd)
        df_canonical = apply_canonical_schema(df_raw, service, row["file_name"], config)
        service_parts.append(df_canonical)
        print(f"  {row['file_name']} -> {n_rows:,} registros canonicos")

    if service_parts:
        from functools import reduce
        from pyspark.sql import DataFrame
        df_service = reduce(DataFrame.union, service_parts)
        canonical_dfs[service] = df_service
        print(f"\n[{service}] Total: {n_rows_total:,} registros | Columnas: {len(df_service.columns)}")
        df_service.printSchema()
    else:
        print(f"[{service}] Sin archivos SUCCESS.")


  yellow_tripdata_2023-01.parquet -> 3066766 registros canónicos
  yellow_tripdata_2023-02.parquet -> 2913955 registros canónicos


KeyboardInterrupt: 

In [ ]:
# Verificacion: mostrar primeras 5 filas de cada servicio canónico
for service, df in canonical_dfs.items():
    print(f"\n=== {service.upper()} — primeras 5 filas ===")
    df.select(
        "trip_id", "service_type", "vendor_id",
        "pickup_datetime", "dropoff_datetime",
        "trip_distance", "fare_amount", "total_amount", "quality_status"
    ).show(5, truncate=40)

### Clasificación de archivos dañados → data/quarantine/

Los archivos con `read_status` en CORRUPT, EMPTY o SCHEMA_ERROR se clasifican en una de las 7 categorías de quarantine y se genera un JSON de trazabilidad por cada uno.

In [ ]:
from pathlib import Path

quarantine_dir = str(Path(config["paths"]["quarantine"]))
problem_files = (
    inventory_df
    .filter("read_status IN ('CORRUPT', 'EMPTY', 'SCHEMA_ERROR')")
    .select("file_name", "file_path", "read_status", "error_message")
    .collect()
)

quarantine_records = []
for row in problem_files:
    classification = classify_corrupt_file(row["file_path"], row["error_message"] or "")
    record = generate_quarantine_record(
        file_name=row["file_name"],
        file_path=row["file_path"],
        classification=classification,
        exception_text=row["error_message"] or f"read_status={row['read_status']}",
        phase="EXTRACT",
        quarantine_dir=quarantine_dir,
    )
    quarantine_records.append(record)
    print(f"  {row['file_name']:45s} -> {classification}")

print(f"\nTotal archivos en quarantine: {len(quarantine_records)}")
import os
print("JSONs generados en:", quarantine_dir)
print("Archivos:", os.listdir(quarantine_dir))

### Escritura de data/bronze/ — particionado por service_type/year/month

Se unen los tres DataFrames canónicos y se escribe el resultado en formato Parquet particionado.

In [ ]:
import shutil
import pandas as pd
from pathlib import Path

bronze_path = str(PROJECT_ROOT / config["paths"]["bronze"])
bronze_path_dir = Path(bronze_path)
if bronze_path_dir.exists():
    shutil.rmtree(str(bronze_path_dir))
bronze_path_dir.mkdir(parents=True, exist_ok=True)

print(f"Escribiendo bronze en: {bronze_path}")
total_written = 0
for service, df_spark in canonical_dfs.items():
    df_pd = df_spark.toPandas()
    for (yr, mo), grp in df_pd.groupby(["year", "month"]):
        part_dir = bronze_path_dir / f"service_type={service}" / f"year={int(yr)}" / f"month={int(mo)}"
        part_dir.mkdir(parents=True, exist_ok=True)
        grp.to_parquet(str(part_dir / "part-00000.parquet"), index=False, engine="pyarrow")
        total_written += len(grp)

print(f"Bronze escrito correctamente. {total_written:,} filas en {bronze_path_dir}")


In [ ]:
from pathlib import Path
import pandas as pd

pq_files = sorted(Path(bronze_path).rglob("*.parquet"))
print(f"Archivos parquet en bronze: {len(pq_files)}")

rows_by_svc = {}
col_count = None
col_names = []
for pq_file in pq_files:
    with open(str(pq_file), "rb") as _fh:
        _df = pd.read_parquet(_fh, engine="pyarrow")
    svc_part = next((p for p in pq_file.parts if p.startswith("service_type=")), None)
    svc = svc_part.split("=")[1] if svc_part else pq_file.parent.parent.parent.name
    rows_by_svc[svc] = rows_by_svc.get(svc, 0) + len(_df)
    if col_count is None:
        col_count = len(_df.columns)
        col_names = list(_df.columns)
    del _df

print(f"Columnas: {col_count} (debe ser 24)")
print(f"Campos: {col_names}")
print()
for svc, cnt in sorted(rows_by_svc.items()):
    print(f"  {svc}: {cnt:,} registros")
